In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t060/task.json'
with open(task_file_path, 'r') as file:
    task_data = json.load(file)

# Extract relevant information from the task data
alert_title = task_data.get('alert_title', None)
alert_window = task_data.get('alert_window', None)
alert_entity = task_data.get('alert_entity', None)

alert_title, alert_window, alert_entity

```
Out[1]:
```
The alert is titled "product-catalog 流量下跌告警" (product-catalog traffic drop alert). It occurred within the time window from 2026-04-13 03:37:52 to 2026-04-13 03:42:32 (UTC+8). The entity that triggered the alert is the operation `product-catalog::oteldemo.ProductCatalogService/GetProduct` of type `apm.operation` in the `apm` domain.

The original code execution output of IPython Kernel is also provided below for reference:

('product-catalog 流量下跌告警', {'start': '2026-04-13T03:37:52+08:00', 'end': '2026-04-13T03:42:32+08:00'}, {'entity_id': '1948ddf615871ac6fc18d01f72817e86', 'entity_name': 'product-catalog::oteldemo.ProductCatalogService/GetProduct', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[2]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[3]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[3]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[4]:

import pandas as pd

# Load the metrics.parquet file to explore available KPIs
metrics_file_path = 'rca100/cases/t060/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# Display the first few rows to understand the structure and available KPIs
metrics_data.head()

```
Out[4]:
```
The `metrics.parquet` file was successfully loaded, and it contains telemetry data with columns such as `time`, `domain`, `entity_set`, `entity_id`, `entity_name`, `metric`, `value`, `metric_set_id`, and `service`. The data includes various metrics like `node_not_ready_status` for Kubernetes nodes. This structure will help identify relevant KPIs for the `product-catalog` service and calculate global thresholds for anomaly detection.

The original code execution output of IPython Kernel is also provided below for reference:

time domain entity_set entity_id entity_name                 metric  value                                     metric_set_id service
0  1776019072000000    k8s   k8s.node                        node_not_ready_status    0.0  k8s@metric_set@k8s.metric.high_level_metric_node    None
1  1776019102000000    k8s   k8s.node                        node_not_ready_status    0.0  k8s@metric_set@k8s.metric.high_level_metric_node    None
2  1776019132000000    k8s   k8s.node                        node_not_ready_status    0.0  k8s@metric_set@k8s.metric.high_level_metric_node    None
3  1776019162000000    k8s   k8s.node                        node_not_ready_status    0.0  k8s@metric_set@k8s.metric.high_level_metric_node    None
4  1776019192000000    k8s   k8s.node                        node_not_ready_status    0.0  k8s@metric_set@k8s.metric.high_level_metric_node    None```
```

In [ ]:
# In[5]:

# Filter metrics data for the 'product-catalog' service
product_catalog_metrics = metrics_data[metrics_data['entity_name'].str.contains('product-catalog', case=False, na=False)]

# Create a new column to classify time series by 'component-KPI'
product_catalog_metrics['component_KPI'] = product_catalog_metrics['entity_name'] + '-' + product_catalog_metrics['metric']

# Aggregate the data to create time series for each 'component-KPI'
product_catalog_time_series = product_catalog_metrics.groupby(['component_KPI', 'time'])['value'].sum().reset_index()

product_catalog_time_series.head()

```
Out[5]:
```
The metrics data for the `product-catalog` service has been successfully aggregated into time series, classified by 'component-KPI' (e.g., `product-catalog-avg_request_latency`). Each time series includes timestamps (`time`) and corresponding metric values (`value`), enabling further analysis of the service's performance and anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

component_KPI              time     value
0  product-catalog-79f7b8c4b-lmp9q-avg_request_la...  1776019072000000  0.000063
1  product-catalog-79f7b8c4b-lmp9q-avg_request_la...  1776019102000000  0.000064
2  product-catalog-79f7b8c4b-lmp9q-avg_request_la...  1776019132000000  0.000065
3  product-catalog-79f7b8c4b-lmp9q-avg_request_la...  1776019162000000  0.000062
4  product-catalog-79f7b8c4b-lmp9q-avg_request_la...  1776019192000000  0.000060```
```

In [ ]:
# In[6]:

# Calculate global thresholds (e.g., P95) for each 'component-KPI'
global_thresholds = product_catalog_time_series.groupby('component_KPI')['value'].quantile(0.95).reset_index()
global_thresholds.rename(columns={'value': 'P95_threshold'}, inplace=True)

global_thresholds

```
Out[6]:
```
The global P95 thresholds have been calculated for each 'component-KPI' time series of the `product-catalog` service. These thresholds represent the 95th percentile of metric values, which can be used to identify anomalies during the alert time window. Examples include thresholds for `avg_request_latency`, `request_count`, `cpu`, `latency`, `mem`, and `workload`.

The original code execution output of IPython Kernel is also provided below for reference:

component_KPI  P95_threshold
0   product-catalog-79f7b8c4b-lmp9q-avg_request_la...   1.047451e-04
1       product-catalog-79f7b8c4b-lmp9q-request_count   4.521550e+03
2   product-catalog-79f7b8c4b-rmpgh-avg_request_la...   7.492085e-05
3       product-catalog-79f7b8c4b-rmpgh-request_count   5.992900e+03
4                                 product-catalog-cpu   1.403562e-02
5                             product-catalog-latency   8.719051e-05
6                                 product-catalog-mem   4.178958e+07
7                            product-catalog-workload   9.769550e+03
8   product-catalog::oteldemo.ProductCatalogServic...   9.187916e-05
9   product-catalog::oteldemo.ProductCatalogServic...   6.575000e+03
10  product-catalog::oteldemo.ProductCatalogServic...   8.794898e-05
11  product-catalog::oteldemo.ProductCatalogServic...   3.237800e+03```
```

In [ ]:
# In[7]:

import pytz
from datetime import datetime

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-04-13T03:37:52+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-04-13T03:42:32+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the alert time window to microseconds for filtering
alert_start_us = int(alert_start.timestamp() * 1_000_000)
alert_end_us = int(alert_end.timestamp() * 1_000_000)

# Filter the aggregated metrics data for the alert time window
filtered_metrics = product_catalog_time_series[
    (product_catalog_time_series['time'] >= alert_start_us) &
    (product_catalog_time_series['time'] <= alert_end_us)
]

# Merge with global thresholds to identify anomalies
filtered_metrics = filtered_metrics.merge(global_thresholds, on='component_KPI', how='left')
filtered_metrics['is_anomaly'] = filtered_metrics['value'] > filtered_metrics['P95_threshold']

filtered_metrics

```
Out[7]:
```
The metrics data for the `product-catalog` service during the alert time window (2026-04-13 03:37:52 to 2026-04-13 03:42:32 UTC+8) was successfully filtered and compared against the global P95 thresholds. No anomalies were detected, as all metric values were below their respective P95 thresholds.

The original code execution output of IPython Kernel is also provided below for reference:

component_KPI              time       value  P95_threshold  is_anomaly
0    product-catalog-79f7b8c4b-lmp9q-avg_request_la...  1776022672000000    0.000051       0.000105       False
1    product-catalog-79f7b8c4b-lmp9q-avg_request_la...  1776022702000000    0.000071       0.000105       False
2    product-catalog-79f7b8c4b-lmp9q-avg_request_la...  1776022732000000    0.000090       0.000105       False
3    product-catalog-79f7b8c4b-lmp9q-avg_request_la...  1776022762000000    0.000101       0.000105       False
4    product-catalog-79f7b8c4b-lmp9q-avg_request_la...  1776022792000000    0.000102       0.000105       False
..                                                 ...               ...         ...            ...         ...
113  product-catalog::oteldemo.ProductCatalogServic...  1776022822000000  585.000000    3237.800000       False
114  product-catalog::oteldemo.ProductCatalogServic...  1776022852000000  586.000000    3237.800000       False
115  product-catalog::oteldemo.ProductCatalogServic...  1776022882000000  550.000000    3237.800000       False
116  product-catalog::oteldemo.ProductCatalogServic...  1776022912000000  520.000000    3237.800000       False
117  product-catalog::oteldemo.ProductCatalogServic...  1776022942000000  545.000000    3237.800000       False

[118 rows x 5 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t060/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Convert relevant columns to numeric for filtering
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')
traces_data['duration'] = pd.to_numeric(traces_data['duration'], errors='coerce')

# Filter traces for the specific operation and alert time window
filtered_traces = traces_data[
    (traces_data['spanName'] == 'product-catalog::oteldemo.ProductCatalogService/GetProduct') &
    (traces_data['startTime'] >= alert_start_us * 1_000) &  # Convert microseconds to nanoseconds
    (traces_data['startTime'] <= alert_end_us * 1_000)
]

# Look for error traces or unusually long durations
error_traces = filtered_traces[filtered_traces['statusCode'] != 0]
slow_traces = filtered_traces.sort_values(by='duration', ascending=False).head(10)

error_traces, slow_traces

```
Out[8]:
```
No error traces or unusually long-duration traces were found for the `product-catalog::oteldemo.ProductCatalogService/GetProduct` operation during the alert time window (2026-04-13 03:37:52 to 2026-04-13 03:42:32, UTC+8). This indicates no trace-level issues were detected for this operation during the specified period.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, events, ip, links]
Index: []

[0 rows x 21 columns], Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, events, ip, links]
Index: []

[0 rows x 21 columns])```
```

In [ ]:
# In[9]:

# Load the logs.parquet file
logs_file_path = 'rca100/cases/t060/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Convert the log timestamp column to numeric for filtering
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs for the 'product-catalog' service during the alert time window
filtered_logs = logs_data[
    (logs_data['_time_'] >= alert_start_us) &
    (logs_data['_time_'] <= alert_end_us) &
    (logs_data['_container_name_'].str.contains('product-catalog', case=False, na=False))
]

# Look for error messages, warnings, or unusual patterns in the log content
error_logs = filtered_logs[filtered_logs['content'].str.contains('error|warn|fail', case=False, na=False)]

error_logs

```
Out[9]:
```
No error messages, warnings, or unusual patterns were found in the logs related to the `product-catalog` service during the alert time window (2026-04-13 03:37:52 to 2026-04-13 03:42:32, UTC+8). This indicates no log-level issues were detected for this service during the specified period.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[10]:

# Filter metrics data for interactions or dependencies related to the 'product-catalog' service
interaction_metrics = metrics_data[
    (metrics_data['metric'].str.contains('calls|request_count', case=False, na=False)) &
    (metrics_data['entity_name'].str.contains('product-catalog', case=False, na=False))
]

# Identify upstream services or external dependencies by grouping and analyzing entity names
upstream_dependencies = interaction_metrics.groupby('entity_name')['value'].sum().reset_index()

upstream_dependencies

```
Out[10]:
```
The analysis of metrics data indicates that the `product-catalog` service has interactions or dependencies involving its pods (`product-catalog-79f7b8c4b-lmp9q` and `product-catalog-79f7b8c4b-rmpgh`) and specific operations (`product-catalog::oteldemo.ProductCatalogService/GetProduct` and `product-catalog::oteldemo.ProductCatalogService/ListProducts`). These entities show significant request counts, suggesting they are key components or dependencies within the service.

The original code execution output of IPython Kernel is also provided below for reference:

entity_name     value
0                    product-catalog-79f7b8c4b-lmp9q  210855.0
1                    product-catalog-79f7b8c4b-rmpgh  282169.0
2  product-catalog::oteldemo.ProductCatalogServic...  334526.0
3  product-catalog::oteldemo.ProductCatalogServic...  158498.0```
```

In [ ]:
# In[11]:

# Filter metrics data for upstream services or dependencies of the 'product-catalog' service
upstream_metrics = metrics_data[
    (metrics_data['entity_name'].isin(upstream_dependencies['entity_name'])) &
    (metrics_data['metric'].isin(['request_count', 'latency', 'error_rate']))
]

# Calculate global thresholds (e.g., P95) for each 'component-KPI' of upstream entities
upstream_metrics['component_KPI'] = upstream_metrics['entity_name'] + '-' + upstream_metrics['metric']
upstream_global_thresholds = upstream_metrics.groupby('component_KPI')['value'].quantile(0.95).reset_index()
upstream_global_thresholds.rename(columns={'value': 'P95_threshold'}, inplace=True)

# Filter upstream metrics for the alert time window
filtered_upstream_metrics = upstream_metrics[
    (upstream_metrics['time'] >= alert_start_us) &
    (upstream_metrics['time'] <= alert_end_us)
]

# Merge with global thresholds to identify anomalies
filtered_upstream_metrics = filtered_upstream_metrics.merge(upstream_global_thresholds, on='component_KPI', how='left')
filtered_upstream_metrics['is_anomaly'] = filtered_upstream_metrics['value'] > filtered_upstream_metrics['P95_threshold']

filtered_upstream_metrics

```
Out[11]:
```
The metrics data for upstream services and dependencies of the `product-catalog` service during the alert time window (2026-04-13 03:37:52 to 2026-04-13 03:42:32, UTC+8) was analyzed. Metrics such as `request_count`, `latency`, and `error_rate` were checked against their global P95 thresholds, and no anomalies were detected. All metric values for the upstream entities were below their respective thresholds, indicating normal behavior during the specified period.

The original code execution output of IPython Kernel is also provided below for reference:

time domain     entity_set                         entity_id                                        entity_name  ...                            metric_set_id  service                                      component_KPI P95_threshold is_anomaly
0   1776022672000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       3237.80      False
1   1776022732000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       3237.80      False
2   1776022762000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       3237.80      False
3   1776022792000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       3237.80      False
4   1776022822000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       3237.80      False
5   1776022852000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       3237.80      False
6   1776022882000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       3237.80      False
7   1776022912000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       3237.80      False
8   1776022942000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       3237.80      False
9   1776022672000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       6575.00      False
10  1776022702000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       6575.00      False
11  1776022732000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       6575.00      False
12  1776022762000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       6575.00      False
13  1776022792000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       6575.00      False
14  1776022822000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       6575.00      False
15  1776022852000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       6575.00      False
16  1776022882000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       6575.00      False
17  1776022912000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       6575.00      False
18  1776022942000000    apm  apm.operation  1948ddf615871ac6fc18d01f72817e86  product-catalog::oteldemo.ProductCatalogServic...  ...  apm@metric_set@apm.metric.apm.operation     None  product-catalog::oteldemo.ProductCatalogServic...       6575.00      False
19  1776022672000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-rmpgh-request_count       5992.90      False
20  1776022702000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-rmpgh-request_count       5992.90      False
21  1776022732000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-rmpgh-request_count       5992.90      False
22  1776022762000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-rmpgh-request_count       5992.90      False
23  1776022792000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-rmpgh-request_count       5992.90      False
24  1776022822000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-rmpgh-request_count       5992.90      False
25  1776022852000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-rmpgh-request_count       5992.90      False
26  1776022882000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-rmpgh-request_count       5992.90      False
27  1776022912000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-rmpgh-request_count       5992.90      False
28  1776022942000000    apm   apm.instance  df6348543e3360ea1206bd20179899fc                    product-catalog-79f7b8c4b-rmpgh  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-rmpgh-request_count       5992.90      False
29  1776022672000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-lmp9q-request_count       4521.55      False
30  1776022702000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-lmp9q-request_count       4521.55      False
31  1776022732000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-lmp9q-request_count       4521.55      False
32  1776022762000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-lmp9q-request_count       4521.55      False
33  1776022792000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-lmp9q-request_count       4521.55      False
34  1776022822000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-lmp9q-request_count       4521.55      False
35  1776022852000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-lmp9q-request_count       4521.55      False
36  1776022882000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-lmp9q-request_count       4521.55      False
37  1776022912000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-lmp9q-request_count       4521.55      False
38  1776022942000000    apm   apm.instance  34e3e1c865aacce7dbda259b7ca3cefc                    product-catalog-79f7b8c4b-lmp9q  ...   apm@metric_set@apm.metric.apm.instance     None      product-catalog-79f7b8c4b-lmp9q-request_count       4521.55      False

[39 rows x 12 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[12]:

# Load the events.parquet file
events_file_path = 'rca100/cases/t060/events.parquet'
events_data = pd.read_parquet(events_file_path)

# Check the column names to identify the correct timestamp column
events_data.columns

# Display the first few rows to understand the structure
events_data.head()

```
Out[12]:
```
The `events.parquet` file contains Kubernetes events with columns such as `eventId`, `hostname`, `level`, `clusterId`, `clusterName`, `pod_id`, and `pod_name`. These events include both `Normal` and `Warning` levels, and some events are related to specific pods (e.g., `prometheus-7db7ccbd6-fzwnl`). This structure will help identify infrastructure issues, such as node failures, pod scheduling problems, or network issues, during the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

eventId               hostname    level                          clusterId                clusterName __topic__                                pod_id                    pod_name
0  {\n "metadata": {\n  "name": "prometheus-7db7c...  cn-hongkong.10.0.2.37   Normal  cfbbc0eabc19d43c0a6fb6889b4451ad0  cms-otel-demo-cn-hongkong            3447490d-23c8-40f9-bb68-877228691033  prometheus-7db7ccbd6-fzwnl
1  {\n "metadata": {\n  "name": "ack-sysom-monito...                         Warning  cfbbc0eabc19d43c0a6fb6889b4451ad0  cms-otel-demo-cn-hongkong                                            None                        None
2  {\n "metadata": {\n  "name": "ack-sysom-monito...                          Normal  cfbbc0eabc19d43c0a6fb6889b4451ad0  cms-otel-demo-cn-hongkong                                            None                        None
3  {\n "metadata": {\n  "name": "ack-sysom-monito...                          Normal  cfbbc0eabc19d43c0a6fb6889b4451ad0  cms-otel-demo-cn-hongkong                                            None                        None
4  {\n "metadata": {\n  "name": "ack-node-problem...                         Warning  cfbbc0eabc19d43c0a6fb6889b4451ad0  cms-otel-demo-cn-hongkong                                            None                        None```
```

In [ ]:
# In[13]:

# Convert the event timestamp to numeric for filtering
events_data['event_time'] = pd.to_numeric(events_data['eventId'].apply(lambda x: json.loads(x).get('metadata', {}).get('creationTimestamp', 0)), errors='coerce')

# Filter events for the alert time window
filtered_events = events_data[
    (events_data['event_time'] >= alert_start_us) &
    (events_data['event_time'] <= alert_end_us) &
    (events_data['level'] == 'Warning') &
    (events_data['pod_name'].str.contains('product-catalog', case=False, na=False))
]

# Analyze the event messages for indications of infrastructure issues
filtered_events['event_message'] = events_data['eventId'].apply(lambda x: json.loads(x).get('message', ''))

filtered_events[['event_time', 'hostname', 'level', 'pod_name', 'event_message']]

```
Out[13]:
```
The filtered Kubernetes events during the alert time window include several `Warning` level events, but none are specifically related to the `product-catalog` service or its pods. The events include messages about container image issues, failed daemon pods, pod creation/deletion, readiness probe failures, and OOM kills on a node (`cn-hongkong.10.0.2.37`). These events suggest potential infrastructure issues, but none directly implicate the `product-catalog` service.

The original code execution output of IPython Kernel is also provided below for reference:

event_time hostname level pod_name                                      event_message
0           NaN      NaN   NaN      NaN  Container image "o11y-demo-registry-vpc.cn-hon...
1           NaN      NaN   NaN      NaN  Found failed daemon pod kube-system/ack-sysom-...
2           NaN      NaN   NaN      NaN               Deleted pod: ack-sysom-monitor-2dcrr
3           NaN      NaN   NaN      NaN               Created pod: ack-sysom-monitor-x5njg
4           NaN      NaN   NaN      NaN  Found failed daemon pod kube-system/ack-node-p...
..          ...      ...   ...      ...                                                ...
98          NaN      NaN   NaN      NaN                   Created pod: node-exporter-gjm2n
99          NaN      NaN   NaN      NaN  Container image "o11y-demo-registry-vpc.cn-hon...
100         NaN      NaN   NaN      NaN            Attempting to reclaim ephemeral-storage
101         NaN      NaN   NaN      NaN  Readiness probe failed: HTTP probe failed with...
102         NaN      NaN   NaN      NaN  pod was OOM killed. node:cn-hongkong.10.0.2.37...

[103 rows x 5 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[14]:

# Filter metrics data for node-level metrics related to the node 'cn-hongkong.10.0.2.37'
node_metrics = metrics_data[
    (metrics_data['entity_name'] == 'cn-hongkong.10.0.2.37') &
    (metrics_data['metric'].isin(['node_memory_usage_rate', 'node_cpu_usage_rate']))
]

# Calculate global thresholds (e.g., P95) for each 'component-KPI' of the node
node_metrics['component_KPI'] = node_metrics['entity_name'] + '-' + node_metrics['metric']
node_global_thresholds = node_metrics.groupby('component_KPI')['value'].quantile(0.95).reset_index()
node_global_thresholds.rename(columns={'value': 'P95_threshold'}, inplace=True)

# Filter node metrics for the alert time window
filtered_node_metrics = node_metrics[
    (node_metrics['time'] >= alert_start_us) &
    (node_metrics['time'] <= alert_end_us)
]

# Merge with global thresholds to identify anomalies
filtered_node_metrics = filtered_node_metrics.merge(node_global_thresholds, on='component_KPI', how='left')
filtered_node_metrics['is_anomaly'] = filtered_node_metrics['value'] > filtered_node_metrics['P95_threshold']

filtered_node_metrics

```
Out[14]:
```
No node-level metrics (`node_memory_usage_rate`, `node_cpu_usage_rate`) were found for the node `cn-hongkong.10.0.2.37` during the alert time window (2026-04-13 03:37:52 to 2026-04-13 03:42:32, UTC+8). Consequently, no anomalies could be identified for this node based on the metrics data.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [time, domain, entity_set, entity_id, entity_name, metric, value, metric_set_id, service, component_KPI, P95_threshold, is_anomaly]
Index: []

[0 rows x 12 columns]```
```